In [19]:
from catboost import CatBoostRegressor
import mlflow
import numpy as np
import optuna
from optuna_integration.catboost import CatBoostPruningCallback
import pandas as pd

from restaurant_visitor_eda.config import PROCESSED_DATA_DIR
from restaurant_visitor_eda.features import (
    binary_features,
    categorical_features,
    get_custom_cv_splits,
    numeric_features,
)

In [20]:
def add_store_age_features(
    df_train: pd.DataFrame, df_test: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    first_open_dates = df_train.groupby("air_store_id")["visit_date"].min().reset_index()
    first_open_dates.columns = ["air_store_id", "first_open_date"]

    df_train = pd.merge(df_train, first_open_dates, on="air_store_id", how="left")
    df_test = pd.merge(df_test, first_open_dates, on="air_store_id", how="left")

    df_train["days_since_first_open"] = (
        df_train["visit_date"] - df_train["first_open_date"]
    ).dt.days
    df_test["days_since_first_open"] = (df_test["visit_date"] - df_test["first_open_date"]).dt.days

    df_train.drop(columns=["first_open_date"], inplace=True)
    df_test.drop(columns=["first_open_date"], inplace=True)

    return df_train, df_test


def compute_calendar_lags_hierarchical(df: pd.DataFrame, lags: list[int]) -> pd.DataFrame:
    res_df = df.copy()

    for lag in lags:
        shifted = df[["air_store_id", "visit_date", "visitors"]].copy()
        shifted["visit_date"] = shifted["visit_date"] + pd.to_timedelta(lag, unit="D")
        shifted = shifted.rename(columns={"visitors": f"lag_{lag}"})

        res_df = pd.merge(res_df, shifted, on=["air_store_id", "visit_date"], how="left")

        res_df[f"lag_{lag}"] = (
            res_df[f"lag_{lag}"]
            .fillna(res_df["store_dow_mean_cum"])
            .fillna(res_df["store_mean_cum"])
            .fillna(res_df["genre_geo_mean_cum"])
        )

    return res_df

In [21]:
df_train = pd.read_csv(PROCESSED_DATA_DIR / "train_features.csv", parse_dates=["visit_date"])
df_test = pd.read_csv(PROCESSED_DATA_DIR / "test_features.csv", parse_dates=["visit_date"])

print(f"Base Train shape: {df_train.shape}")
print(f"Base Test shape: {df_test.shape}")

df_train, df_test = add_store_age_features(df_train, df_test)

df_train["is_test"] = 0
df_test["is_test"] = 1

df_all = pd.concat([df_train, df_test], ignore_index=True)
df_all = df_all.sort_values(["air_store_id", "visit_date"]).reset_index(drop=True)

df_all = compute_calendar_lags_hierarchical(df_all, lags=[39, 42])

df_train_final = df_all[df_all["is_test"] == 0].copy().reset_index(drop=True)
df_test_final = df_all[df_all["is_test"] == 1].copy().reset_index(drop=True)

df_train_final.drop(columns=["is_test"], inplace=True)
df_test_final.drop(columns=["is_test", "visitors"], inplace=True)

print(f"Engineered Train shape: {df_train_final.shape}")
print(f"Engineered Test shape: {df_test_final.shape}")

Base Train shape: (252108, 28)
Base Test shape: (32019, 28)
Engineered Train shape: (252108, 31)
Engineered Test shape: (32019, 30)


In [22]:
local_categorical_features = categorical_features.copy()
local_binary_features = binary_features.copy()

local_numeric_features = numeric_features.copy()
local_numeric_features.extend(["days_since_first_open", "lag_39", "lag_42"])

features = local_categorical_features + local_numeric_features + local_binary_features

X_full = df_train_final[features]
y_full = np.log1p(df_train_final["visitors"].values)

cv_splits = get_custom_cv_splits(df_train_final, n_splits=3, val_days=39)

Fold 1: Train ends 2017-03-14| Val: 2017-03-15 to 2017-04-22
Fold 2: Train ends 2017-02-03| Val: 2017-02-04 to 2017-03-14
Fold 3: Train ends 2016-12-26| Val: 2016-12-27 to 2017-02-03


In [23]:
def objective(trial: optuna.trial.Trial) -> float:
    params = {
        "iterations": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 9, 12),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 25.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.1, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "random_seed": 42,
        "od_type": "Iter",
        "od_wait": 50,
        "verbose": False,
    }

    pruning_callback = CatBoostPruningCallback(trial, "RMSE")
    cv_scores = []
    fold_iters = []

    for fold, (train_idx, val_idx) in enumerate(cv_splits):
        X_train, y_train = X_full.iloc[train_idx], y_full[train_idx]
        X_val, y_val = X_full.iloc[val_idx], y_full[val_idx]

        model = CatBoostRegressor(**params, cat_features=local_categorical_features)

        if fold == 0:
            model.fit(X_train, y_train, eval_set=(X_val, y_val), callbacks=[pruning_callback])
            pruning_callback.check_pruned()
        else:
            model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=100)

        best_score = model.get_best_score()["validation"]["RMSE"]
        best_iter = model.get_best_iteration()

        cv_scores.append(best_score)
        fold_iters.append(best_iter)

    trial.set_user_attr("mean_best_iter", int(np.mean(fold_iters)))

    return np.mean(cv_scores)

In [24]:
mlflow.set_tracking_uri("sqlite:///mlflow_tracking.db")
mlflow.set_experiment("CatBoost_Optuna_Tuning_With_Lags")

with mlflow.start_run(run_name="optuna_search_cold_start_mitigation"):
    pruner = optuna.pruners.MedianPruner(n_warmup_steps=50)
    study = optuna.create_study(direction="minimize", pruner=pruner)

    study.optimize(objective, n_trials=50, show_progress_bar=True)

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_cv_rmse", study.best_value)
    mlflow.log_metric("mean_best_iter", study.best_trial.user_attrs["mean_best_iter"])

    print(f"\n Best RMSE with lags and age: {study.best_value:.4f}")

[I 2026-06-29 11:21:51,968] A new study created in memory with name: no-name-0e44f86b-90a4-404a-85c8-278400ad77d3


  0%|          | 0/50 [00:00<?, ?it/s]

/tmp/ipykernel_6476/850712039.py:17: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_callback = CatBoostPruningCallback(trial, "RMSE")


[I 2026-06-29 11:24:55,170] Trial 0 finished with value: 0.5052715415391538 and parameters: {'learning_rate': 0.03184613847031136, 'depth': 10, 'l2_leaf_reg': 3.262494053515331, 'random_strength': 1.4376523028240509, 'bagging_temperature': 0.5908488515161102}. Best is trial 0 with value: 0.5052715415391538.
[I 2026-06-29 11:26:29,749] Trial 1 finished with value: 0.5040392889672473 and parameters: {'learning_rate': 0.08837410942528029, 'depth': 10, 'l2_leaf_reg': 13.60222820718246, 'random_strength': 0.10428252064115963, 'bagging_temperature': 0.12241732077285938}. Best is trial 1 with value: 0.5040392889672473.
[I 2026-06-29 11:28:15,690] Trial 2 finished with value: 0.5043313245848194 and parameters: {'learning_rate': 0.057454889165739806, 'depth': 10, 'l2_leaf_reg': 1.105458002370091, 'random_strength': 0.1430391688537791, 'bagging_temperature': 0.9880611721464574}. Best is trial 1 with value: 0.5040392889672473.
[I 2026-06-29 11:31:40,903] Trial 3 finished with value: 0.50437602049

In [25]:
print("\n--- BEST PARAMS ---")
for key, value in study.best_params.items():
    print(f"{key}: {value}")
print(f"\n Best RMSLE during CV: {study.best_value:.4f}")

optimal_iterations = study.best_trial.user_attrs["mean_best_iter"]
print(f"Optimal Iterations: {optimal_iterations}")


--- BEST PARAMS ---
learning_rate: 0.08894248144217208
depth: 12
l2_leaf_reg: 5.086482431213367
random_strength: 0.13766952070768307
bagging_temperature: 0.6422531882429172

 Best RMSLE during CV: 0.5019
Optimal Iterations: 109


In [35]:
final_params = study.best_params.copy()
final_params["iterations"] = int(optimal_iterations * 1.5)
final_params["loss_function"] = "RMSE"
final_params["eval_metric"] = "RMSE"
final_params["random_seed"] = 42
final_params["learning_rate"] = final_params["learning_rate"] / 1.5

final_model = CatBoostRegressor(**final_params, cat_features=local_categorical_features)
final_model.fit(X_full, y_full, verbose=100)

0:	learn: 0.7818690	total: 60.3ms	remaining: 9.76s
100:	learn: 0.5021371	total: 7.9s	remaining: 4.85s
162:	learn: 0.4923882	total: 14.7s	remaining: 0us


CatBoostRegressor(bagging_temperature=0.6422531882429172, cat_features=['air_store_id', 'air_genre_name', 'day_of_week', 'month', 'day_pattern', 'prefecture', 'district', 'block'], depth=12, eval_metric='RMSE', iterations=163, l2_leaf_reg=5.086482431213367, learning_rate=0.05929498762811472, loss_function='RMSE', random_seed=42, random_strength=0.13766952070768307)

In [36]:
X_test = df_test_final[features]

preds_log = final_model.predict(X_test)
preds_real_clipped = np.clip(np.expm1(preds_log), 1.0, None)

submission = pd.DataFrame(
    {
        "id": df_test_final["air_store_id"]
        + "_"
        + df_test_final["visit_date"].dt.strftime("%Y-%m-%d"),
        "visitors": preds_real_clipped,
    }
)

submission_path = "submission_catboost_optuna_lags_and_age.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")
submission.head()

Submission saved to submission_catboost_optuna_lags_and_age.csv


,id,visitors
0,air_00a91d42b08b08d9_2017-04-23,3.842604
1,air_00a91d42b08b08d9_2017-04-24,22.322568
2,air_00a91d42b08b08d9_2017-04-25,24.652362
3,air_00a91d42b08b08d9_2017-04-26,27.968502
4,air_00a91d42b08b08d9_2017-04-27,29.479316
